# Phase 4 — Cross-Year MIL Robustness Grid

Confirm the diagnostic finding (MIL ≈ mean pool ≈ MIL_shared; rank1 = best for
early phases) across **all 5 LOYO folds**, not just test_year=2020.

15 fits = 3 MIL conditions × 5 LOYO folds × 1 stream (All). Estimated ~2 hours total.

Conditions:
- `MIL_resolved`  — phase-resolved attention
- `MIL_shared`    — phase-shared attention
- `mean_pool`     — uniform attention (no learning)

`rank1` is not run here — the Phase 3 LSTM already provides those numbers,
giving us a 4-way comparison via merging.

Resumable: each fold's results are saved as it completes.


## 1. Setup

In [ ]:
from pathlib import Path
import sys, os, time, gc, psutil
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
OUTPUT_DIR  = PROJECT_ROOT / 'phase4_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS       = [2017, 2018, 2019, 2020, 2021]
EPOCHS      = 15
HIDDEN      = 32
BATCH_SIZE  = 8
LR          = 1e-3
USE_AMP     = True


## 2. Helpers

In [ ]:
from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)
from phase3_baselines.evaluate import evaluate_predictions
from phase3_baselines.resumable import append_row, already_done, save_pred_npz
from phase4_mil.bag_dataset import build_bags
from phase4_mil.train import train_mil_fold, STREAM_CONFIGS
from phase4_mil import model as _model_mod
from phase4_mil import train as _train_mod
from phase4_mil.model import MILModel


def load_loyo_bags(test_year, use_rank1=False):
    policy = SelectionPolicy(use_only_rank1=use_rank1)
    test_b = apply_selection(
        load_year_bundle(TENSORS_DIR / f'phenology_{test_year}.npz'), policy)
    train_parts = {}
    for y in YEARS:
        if y == test_year: continue
        train_parts[y] = apply_selection(
            load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
    train_b = concat_bundles(train_parts, years=[y for y in YEARS if y != test_year])
    del train_parts; gc.collect()
    return build_bags(train_b), build_bags(test_b)


class MeanPoolModel(MILModel):
    """Uniform attention (no learning)."""
    def forward(self, batch, active=('S1','S2','Met')):
        X = batch['X']; fmask = batch['feat_mask']; bmask = batch['bag_mask']
        h_seq = self.encode_bag(X, fmask, active=active)
        bm = bmask.float()
        counts = bm.sum(dim=1, keepdim=True).clamp(min=1.0)
        alpha_unif = (bm / counts).unsqueeze(-1).expand(-1, -1, self.n_phases)
        z = torch.einsum('bjth,bjp->btph', h_seq, alpha_unif)
        station_logits = torch.einsum('btph,ph->btp', z, self.station_head) + self.station_bias
        field_logits = self.field_head(h_seq)
        return {'station_logits': station_logits, 'field_logits': field_logits,
                'alpha': alpha_unif, 'h_field': h_seq.mean(dim=2)}


def record_metrics(csv_path, condition, stream, test_year, result, phase_ids):
    rep_st = evaluate_predictions(
        pred_days=result['station_pred'].astype(np.float32),
        true_days=result['station_truth'].astype(np.float32),
        phase_ids=phase_ids, years=result['station_year'], ranks=None,
    )
    rep_fd = evaluate_predictions(
        pred_days=result['field_pred'].astype(np.float32),
        true_days=result['field_truth'].astype(np.float32),
        phase_ids=phase_ids, years=result['field_year'],
        ranks=result['field_rank'],
    )
    for phase, m in rep_st['per_phase'].iterrows():
        append_row(csv_path, {'condition': condition, 'stream': stream,
                                'test_year': int(test_year), 'level': 'station',
                                'phase': int(phase), **m.to_dict()})
    for phase, m in rep_fd['per_phase'].iterrows():
        append_row(csv_path, {'condition': condition, 'stream': stream,
                                'test_year': int(test_year), 'level': 'field',
                                'phase': int(phase), **m.to_dict()})


RESULTS_CSV = OUTPUT_DIR / 'mil_cross_year_robustness.csv'
PREDS_NPZ   = OUTPUT_DIR / 'mil_cross_year_predictions.npz'

CONDITIONS = [
    ('MIL_resolved', True,  False),
    ('MIL_shared',   False, False),
    ('mean_pool',    None,  False),   # uses MeanPoolModel
]

done = already_done(RESULTS_CSV, ['condition', 'stream', 'test_year'])
print(f'Done: {len(done)} of {len(CONDITIONS) * len(YEARS)}')


## 3. Run the grid

In [ ]:
# Cache bags per fold to avoid reloading
bag_cache = {}
def get_bags(test_year, use_rank1):
    key = (test_year, use_rank1)
    if key not in bag_cache:
        bag_cache[key] = load_loyo_bags(test_year, use_rank1=use_rank1)
    return bag_cache[key]

pbar = tqdm(total=len(CONDITIONS) * len(YEARS), desc='MIL robustness')
for cond_name, pr_attn, use_r1 in CONDITIONS:
    for test_y in YEARS:
        key = (cond_name, 'All', int(test_y))
        if key in done:
            pbar.update(1); continue
        t0 = time.time()
        train_bags, test_bags = get_bags(test_y, use_rank1=use_r1)

        if cond_name == 'mean_pool':
            _orig = (_model_mod.MILModel, _train_mod.MILModel)
            _model_mod.MILModel = MeanPoolModel
            _train_mod.MILModel = MeanPoolModel
            try:
                result = train_mil_fold(
                    train_bags=train_bags, test_bags=test_bags,
                    active=STREAM_CONFIGS['All'],
                    hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                    phase_resolved_attention=False, field_loss_weight=0.3,
                    use_amp=USE_AMP, device=DEVICE, verbose=False,
                )
            finally:
                _model_mod.MILModel, _train_mod.MILModel = _orig
        else:
            result = train_mil_fold(
                train_bags=train_bags, test_bags=test_bags,
                active=STREAM_CONFIGS['All'],
                hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                phase_resolved_attention=(pr_attn if pr_attn is not None else False),
                field_loss_weight=0.3,
                use_amp=USE_AMP, device=DEVICE, verbose=False,
            )

        dt = time.time() - t0
        phase_ids = train_bags['phase_ids']
        record_metrics(RESULTS_CSV, cond_name, 'All', test_y, result, phase_ids)
        save_pred_npz(PREDS_NPZ,
                      f'{cond_name}__All__{test_y}',
                      st_pred=result['station_pred'].astype(np.int32),
                      st_truth=result['station_truth'].astype(np.int32),
                      st_bag=result['station_bag_id'],
                      fd_pred=result['field_pred'].astype(np.int32),
                      fd_truth=result['field_truth'].astype(np.int32),
                      fd_bag=result['field_bag_id'],
                      fd_rank=result['field_rank'],
                      alpha=result['alpha'].astype(np.float32),
                      alpha_bag=result['alpha_bag_id'])
        pbar.set_postfix({'fold': f'{cond_name}/{test_y}', 'secs': f'{dt:.0f}'})
        pbar.update(1)

pbar.close()
print('\nGrid complete.')


## 4. Summary across years

In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(results)} rows')

# Macro MAE per (condition, level, year)
year_summary = (results.groupby(['condition','level','test_year'])['mae']
                        .mean().round(2).unstack('test_year'))
print('\n=== Per-year macro MAE by (condition, level) ===')
print(year_summary.to_string())


In [ ]:
# Cross-year aggregated
agg = (results.groupby(['condition','level'])
              .agg(macro_mae=('mae','mean'),
                    std_across_phases=('mae','std'),
                    macro_r2 =('r2','mean'),
                    macro_within_5d=('within_5d','mean'))
              .reset_index())
print('\n=== Cross-year aggregate ===')
print(agg.round(3).to_string(index=False))


## 5. Compare to Phase 3 LSTM (which gave us the rank1 numbers)

In [ ]:
p3 = pd.read_csv(PROJECT_ROOT / 'phase3_outputs/lstm/lstm_results.csv')
p3_all = p3[(p3['stream'] == 'All')]

# Macro MAE per (policy)
p3_agg = (p3_all.groupby('policy')
                 .agg(macro_mae=('mae','mean'),
                      macro_r2 =('r2','mean'),
                      macro_within_5d=('within_5d','mean')))
print('=== Phase 3 LSTM (All stream, both policies) ===')
print(p3_agg.round(3).to_string())
print()
print('=== Phase 4 MIL (All stream, station-level only) ===')
print(agg[(agg['condition'].isin(['MIL_resolved','MIL_shared','mean_pool'])) &
            (agg['level']=='station')].round(3).to_string(index=False))

# Build a unified comparison: 4 supervision approaches on station-level eval
all_rows = []
for _, r in p3_agg.iterrows():
    all_rows.append({'approach': f'LSTM_{r.name}', 'macro_mae': round(r['macro_mae'],2),
                      'macro_r2': round(r['macro_r2'],3),
                      'within_5d': round(r['macro_within_5d'],3)})
for _, r in agg[(agg['level']=='station')].iterrows():
    all_rows.append({'approach': r['condition'],
                      'macro_mae': round(r['macro_mae'],2),
                      'macro_r2': round(r['macro_r2'],3),
                      'within_5d': round(r['macro_within_5d'],3)})
unified = pd.DataFrame(all_rows).sort_values('macro_mae')
unified.to_csv(OUTPUT_DIR / 'all_approaches_summary.csv', index=False)
print('\n=== All approaches ranked by macro MAE ===')
print(unified.to_string(index=False))


## 6. Per-phase comparison across all approaches

In [ ]:
# Per-phase MAE for each approach
def stack_per_phase():
    rows = []
    for _, r in p3_all.iterrows():
        rows.append({'approach': f'LSTM_{r["policy"]}',
                      'phase': int(r['phase']),
                      'mae': r['mae']})
    for _, r in results[(results['level']=='station')].iterrows():
        rows.append({'approach': r['condition'],
                      'phase': int(r['phase']),
                      'mae': r['mae']})
    return pd.DataFrame(rows)

df = stack_per_phase()
pivot = df.groupby(['approach','phase'])['mae'].mean().unstack('phase').round(2)
pivot['macro'] = pivot.mean(axis=1).round(2)
pivot = pivot.sort_values('macro')
print('=== Per-phase MAE by approach (station-level, mean across LOYO) ===')
print(pivot.to_string())
pivot.to_csv(OUTPUT_DIR / 'per_phase_all_approaches.csv')


---\n**Done.**